# Landscape-size diagnostic — Frame B (2026-05-30)

**Drill-down, NOT a graduation run.** Tests whether a larger memorized Hopfield
landscape collapses the *consolidation-injected* per-seed variance
(σ_A = 0.157 at landscape=64 vs the frozen floor σ_C ≈ 0.069).

- Pre-committed read: `notes/notes/2026-05-30-landscape-sweep-diagnostic-precommit.md`
- Background / why: `reports/117_frameb_leveldid_feasibility_consolidation_variance.md`
- Runtime ≈ **1.8 GPU-hr** (L=64 ≈ 8 min, L=256 ≈ 32 min, L=512 ≈ 64 min).

**Before running:** set *Runtime → Change runtime type → GPU*. Then run the cells
top-to-bottom; the last code cell prints the pre-committed verdict.


In [ ]:
# 1. Clone + checkout the branch, verify the diagnostic code is present.
import os
from pathlib import Path
REPO_DIR = '/content/Neuro-AI'
BRANCH = 'codex/phase5-prime-bundle-first-scene-memory'
%cd /content
!rm -rf Neuro-AI
!git clone https://github.com/Dypatterson/Neuro-AI.git
%cd Neuro-AI
!git checkout {BRANCH}
!git log --oneline -3
os.chdir(REPO_DIR)
gate0 = Path(REPO_DIR) / 'experiments' / 'gate0_frame_a.py'
read  = Path(REPO_DIR) / 'scripts' / 'landscape_sweep_read.py'
if not gate0.exists():
    raise SystemExit('gate0_frame_a.py missing — push the branch and re-run.')
if '--landscape-size' not in gate0.read_text():
    raise SystemExit('--landscape-size flag missing — wrong branch?')
if not read.exists():
    raise SystemExit('scripts/landscape_sweep_read.py missing — push THIS session first.')
print('Landscape diagnostic code verified on branch.')


In [ ]:
# 2. SMOKE — tiny synthetic run on CUDA: confirms runtime + that --landscape-size works.
import subprocess, sys, os, json
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
smoke_out = Path('reports/landscape_smoke'); smoke_out.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, 'experiments/gate0_frame_a.py',
       '--seeds', '0,1,2', '--corpus-source', 'synthetic', '--vocab-size', '40',
       '--D', '256', '--landscape-size', '16', '--window', '4',
       '--n-test-windows', '24', '--n-train-windows', '48',
       '--n-consolidation-events', '50', '--theta-prime-mode', 'default',
       '--device', 'cuda', '--output-dir', str(smoke_out)]
rc = subprocess.call(cmd); print('smoke exit code:', rc)
if rc != 0:
    raise SystemExit('Gate 0 smoke failed — abort before the real sweep.')
sm = json.load(open(smoke_out / 'gate0_summary.json'))
assert sm['gauge_confirmation_E']['byte_identical_4a'], '4a byte-identity FAILED in smoke'
print('smoke OK; --landscape-size accepted; 4a byte-identical:',
      sm['gauge_confirmation_E']['byte_identical_4a'])


In [ ]:
# 3. THE SWEEP — gate0 n=10 at the recovered op point, landscape ∈ {64, 256, 512}.
#    Single CUDA subprocess per L; parent stays CPU-only. ~1.8 GPU-hr total.
import subprocess, sys, os, json
from pathlib import Path
os.environ['PYTHONPATH'] = '/content/Neuro-AI/src'
BASE = ['--seeds', '0,1,2,3,4,5,6,7,8,9', '--D', '4096', '--window', '8',
        '--n-test-windows', '512', '--n-train-windows', '2048',
        '--K', '5', '--beta', '10', '--n-consolidation-events', '1000',
        '--theta-prime-mode', 'both',
        '--alpha-anti', '0.01', '--repulsion-step-size', '0.05',
        '--lr-pull', '0.1', '--lr-push', '0.05',
        '--corpus-source', 'wikitext', '--wikitext-name', 'wikitext-2-raw-v1',
        '--vocab-cap', '1000', '--device', 'cuda']
root = Path('reports/landscape_2026-05-30')
for L in (64, 256, 512):
    out = root / f'L{L:04d}'; out.mkdir(parents=True, exist_ok=True); log = out / 'run.log'
    cmd = [sys.executable, 'experiments/gate0_frame_a.py', *BASE,
           '--landscape-size', str(L), '--output-dir', str(out)]
    print(f'=== L={L} -> {out} ===', flush=True)
    with log.open('w') as f:
        rc = subprocess.call(cmd, stdout=f, stderr=subprocess.STDOUT)
    print('  exit', rc, flush=True)
    if rc != 0:
        os.system(f'tail -50 {log}'); raise SystemExit(f'L={L} run failed — see log above.')
print('sweep done.')
# persist to Drive (so the summaries survive the session)
try:
    from google.colab import drive; drive.mount('/content/drive'); import shutil
    dst = '/content/drive/MyDrive/neuro-ai/results/landscape_2026-05-30'
    shutil.copytree(str(root), dst, dirs_exist_ok=True); print('copied ->', dst)
except Exception as e:
    print('drive copy skipped:', e)


In [ ]:
# 4. PRE-COMMITTED READ — sigma_A(L), sigma_C(L), mean(A-C)(L) + verdict.
import subprocess, sys
paths = [f'L={L}:reports/landscape_2026-05-30/L{L:04d}/gate0_summary.json' for L in (64, 256, 512)]
subprocess.call([sys.executable, 'scripts/landscape_sweep_read.py', *paths])
print('\nPaste this table + verdict back to the assistant for the full write-up (Report 118).')


## What the verdict means (pre-registered — not post-hoc)

| verdict | meaning | next |
|---|---|---|
| **VARIANCE-REDUCIBLE** (σ_A(512) ≤ 0.10 AND mean(A−C) ≥ 0.020) | a bigger landscape tames the consolidation-injected variance | propose a powered run at the best L (op-point sign-off); level-DiD n drops ~424 → ~100–187, slope worth revisiting |
| **VARIANCE-IRREDUCIBLE** (σ_A(512) ≥ 0.13) | landscape is not the knob | Frame B mechanism / operating-point rethink — not more compute |
| **PARTIAL** (0.10 < σ_A(512) < 0.13) | ambiguous | read the σ_A(L) curve; consider L=1024 or modest L↑ + n↑ |
| **CAPACITY-WALL** (mean(A−C) < 0.015 at any L) | bigger landscape exceeded Hopfield capacity at β=10 and killed the signal | bigger L is off the table even if σ dropped |

σ_A(L=64) **must reproduce ≈ 0.157** (the recovered run) — it's the reproducibility anchor. If it doesn't, stop: environment/repro problem.
